<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png" width="250" style="display: block; margin-left: auto; margin-right: auto;">

# Premier League – Data Preparation and Feature Engineering

## Purpose

This notebook turns the main EDA findings into one clear and reproducible preprocessing workflow. The priority is to understand **what each step does and why it is necessary**.

By the end, we will be able to:

- validate the minimum input-data contract;
- correct reproducible data anomalies;
- build historical (temporal) features before the split;
- create train and test partitions using a temporal split;
- impute, encode, and scale different types of variables;
- assemble all transformations into a scikit-learn pipeline;
- verify the design and prevent data leakage.

```text
Raw data
    ↓
Anomaly correction + validation
    ↓
Historical feature engineering
    ↓
Temporal train / test split
    ↓
Imputation + encoding + scaling
    ↓
Model
```

## 1. Imports, configuration, and paths

Dependencies should be installed from the project environment, not from a `pip install` cell inside the notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# Base configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "FTR"

# Reproducible paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "premier_league_2021_2026.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "premier_league_2021_2026.csv"

## 2. Data loading, anomaly correction, and strict validation

The EDA identified the expected columns and valid ranges. We now convert those observations into executable checks with explicit steps. If the input schema changes, the notebook should stop with an informative message instead of silently ignoring the problem.

In [2]:
df = pd.read_csv(DATA_PATH)

# Convert the date for chronological ordering (critical for the temporal split)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values(by=['Date', 'Time']).reset_index(drop=True)

# ---------------------------------------------------------
# ANOMALY CORRECTION (Reproducible)
# ---------------------------------------------------------
# Correction of the error detected in the EDA: AST (9) > AS (8) for West Ham
# Verified with an official source (ESPN). The real value of AS is 18.
match_key = (
    (df["Date"] == "2021-08-15") & 
    (df["HomeTeam"] == "Newcastle") & 
    (df["AwayTeam"] == "West Ham")
)
df.loc[match_key, "AS"] = 18

# ---------------------------------------------------------
# MANDATORY STRUCTURAL AND SEMANTIC VALIDATIONS
# ---------------------------------------------------------
# 1. 380 matches per season (20 teams * 19 home matchdays * 2)
assert (df.groupby("Season").size() == 380).all(), "Error: Incomplete seasons."

# 2. Valid values in the target variable
assert df[TARGET_COLUMN].isin(["H", "D", "A"]).all(), "Error: Invalid values in FTR."

# 3. Distinct teams in each match
assert (df["HomeTeam"] != df["AwayTeam"]).all(), "Error: Matches with HomeTeam equal to AwayTeam."

# 4. Physical logical constraints (after the correction)
assert (df["HST"].fillna(0) <= df["HS"].fillna(0)).all(), "Error: Shots on target (HST) exceed total shots (HS)."
assert (df["AST"].fillna(0) <= df["AS"].fillna(0)).all(), "Error: Shots on target (AST) exceed total shots (AS)."

# 5. Duplicate detection using a composite key
assert df.duplicated(subset=["Season", "Date", "HomeTeam", "AwayTeam"]).sum() == 0, "Error: Duplicate matches exist."

print(f"Dataset validated and corrected: {df.shape[0]} rows x {df.shape[1]} columns.")

Dataset validated and corrected: 1900 rows x 157 columns.


## 3. Historical feature engineering (pre-split)

Features that use `shift()` or `rolling()` over time must be computed **before** the split, but **grouped by season** to avoid carrying data from May into August (avoiding the issue pointed out by the professor).

In [3]:
def build_historical_features(data: pd.DataFrame) -> pd.DataFrame:
    df_hist = data.copy()
    
    # Example solution for RestDays (rest days)
    # Group by Season and Team so that 3-month summer breaks are not counted.
    
    # 1. Create a stacked (long format) dataset to compute per-team statistics
    home_df = df_hist[['Date', 'Season', 'HomeTeam']].rename(columns={'HomeTeam': 'Team'})
    away_df = df_hist[['Date', 'Season', 'AwayTeam']].rename(columns={'AwayTeam': 'Team'})
    teams_df = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date'])
    
    # 2. Compute days since the last match, respecting season boundaries
    teams_df['LastMatchDate'] = teams_df.groupby(['Team', 'Season'])['Date'].shift(1)
    teams_df['RestDays'] = (teams_df['Date'] - teams_df['LastMatchDate']).dt.days
    
    # 3. Impute the first match of the season with a base rest (e.g. 21 preseason days)
    teams_df['RestDays'] = teams_df['RestDays'].fillna(21.0)
    
    # Merge back into the original dataframe
    df_hist = df_hist.merge(
        teams_df[['Date', 'Team', 'RestDays']], 
        left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left'
    ).rename(columns={'RestDays': 'HomeRestDays'}).drop(columns=['Team'])
    
    df_hist = df_hist.merge(
        teams_df[['Date', 'Team', 'RestDays']], 
        left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left'
    ).rename(columns={'RestDays': 'AwayRestDays'}).drop(columns=['Team'])
    
    return df_hist

df = build_historical_features(df)

## 4. Train / test separation (temporal split)

Rationale: since this is a sequential problem, a random split would cause data leakage. The temporal split ensures the model is evaluated on its ability to generalize into the future.

In [4]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    shuffle=False # STRICTLY FALSE FOR TIME SERIES
)

print(f"Train partition: {X_train.shape[0]} rows (Start: {X_train['Date'].min().date()} -> End: {X_train['Date'].max().date()})")
print(f"Test partition: {X_test.shape[0]} rows (Start: {X_test['Date'].min().date()} -> End: {X_test['Date'].max().date()})")

Train partition: 1520 rows (Start: 2021-08-13 -> End: 2025-05-25)
Test partition: 380 rows (Start: 2025-08-15 -> End: 2026-05-24)


## 5. Row-wise (stateless) feature engineering for the pipeline

The function below is **stateless**: it does not learn medians, category frequencies, or any information from other rows. It only combines values already present in each match record. `FunctionTransformer` will place this same function inside the sklearn pipeline.

In [5]:
def create_stateless_features(data: pd.DataFrame) -> pd.DataFrame:
    """Features that do not depend on other rows and can live inside the pipeline."""
    result = data.copy()
    
    # Family 1: Dates
    result['Month'] = result['Date'].dt.month
    result['DayOfWeek'] = result['Date'].dt.dayofweek
    
    # Family 2: Missing-value indicators (boolean flag)
    # Useful because the missing odds detected in the EDA follow a pattern
    result['Missing_B365'] = result['B365H'].isna().astype(int)
    
    return result

feature_engineering = FunctionTransformer(create_stateless_features, validate=False)

## 6. Feature groups and pipeline configuration

A `ColumnTransformer` lets us apply a different preparation strategy to each variable type.

| Group | Preparation |
|---|---|
| Numeric | Median imputation and standardization |
| Categorical | Most-frequent imputation and one-hot encoding |
| Binary indicators | Pass through unchanged |

Variables such as `HTR`, `HS`, `AS`, `FTHG`, `FTAG`, and `Referee` are not included in these lists, so `remainder="drop"` ensures they never reach the model.

In [6]:
# ATTENTION: The variables HTR, HS, AS, FTHG, FTAG, Referee are explicitly omitted
# from these lists. The remainder="drop" parameter ensures they never reach the model.

numeric_features = [
    "B365H", "B365D", "B365A", # Add the desired odds
    "HomeRestDays", "AwayRestDays"
]

categorical_features = [
    "HomeTeam", "AwayTeam", "Month", "DayOfWeek"
]

binary_features = [
    "Missing_B365"
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("binary", "passthrough", binary_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preparation_pipeline = Pipeline(steps=[
    ("feature_engineering", feature_engineering),
    ("preprocessor", preprocessor),
])

## 7. Structural verification

We only transform `X_train` to validate the pipeline design. These checks verify that no observations were lost, every output value is finite, and no leaky features reached the final output.

In [7]:
# We only transform X_train to validate the pipeline design

X_train_prepared = preparation_pipeline.fit_transform(X_train, y_train)
output_feature_names = preparation_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"Input dimensions: {X_train.shape}")
print(f"Output dimensions: {X_train_prepared.shape}")

# Post-pipeline safety validations
assert X_train_prepared.shape[0] == X_train.shape[0], "Observations were lost in the pipeline."
assert X_train_prepared.shape[1] == len(output_feature_names), "Dimension mismatch in feature names."

# Convert to a dense matrix if sparse for validation
values_to_check = X_train_prepared.toarray() if hasattr(X_train_prepared, "toarray") else X_train_prepared
assert np.isfinite(values_to_check).all(), "NaN or Infinite values remain that are incompatible with the models."

# Confirmation that future features did not pass through the preprocessor
leaky_features = ["HTR", "Referee", "FTHG", "FTAG", "HS", "AS"]
for feature in leaky_features:
    assert feature not in output_feature_names, f"LEAKAGE ALERT! Column {feature} reached the final output."

print("\nAll structural and data-leakage prevention checks passed successfully.")

Input dimensions: (1520, 158)
Output dimensions: (1520, 75)

All structural and data-leakage prevention checks passed successfully.
